# Missingness, Label Noise, and Data Quality Diagnostics Lab


## Purpose

This lab simulates missingness and noisy labels as data-quality failure modes.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 5000

data = pd.DataFrame({
    "unit_id": [f"unit_{i:05d}" for i in range(1, n + 1)],
    "group": rng.choice(["A", "B"], size=n, p=[0.82, 0.18]),
    "feature_signal": rng.normal(0, 1, size=n),
    "measurement_quality": rng.choice(
        ["high", "medium", "low"],
        size=n,
        p=[0.55, 0.30, 0.15],
    ),
})

data["measurement_error_sd"] = np.where(data["group"] == "B", 0.45, 0.20)

data["measured_feature"] = (
    data["feature_signal"]
    + rng.normal(0, data["measurement_error_sd"], size=n)
)

data.head()

In [ ]:
logit = -0.10 + 1.20 * data["feature_signal"] + np.where(data["group"] == "B", -0.15, 0)
probability = 1 / (1 + np.exp(-logit))
data["true_label"] = rng.binomial(1, probability)

label_flip_probability = np.where(
    data["measurement_quality"] == "low",
    0.18,
    np.where(data["measurement_quality"] == "medium", 0.08, 0.03),
)

label_flip = rng.binomial(1, label_flip_probability)
data["observed_label"] = np.where(label_flip == 1, 1 - data["true_label"], data["true_label"])

missing_probability = np.where(data["group"] == "B", 0.20, 0.06)
data["measured_feature_missing"] = rng.binomial(1, missing_probability)

quality_summary = (
    data
    .groupby(["group", "measurement_quality"])
    .agg(
        units=("unit_id", "count"),
        missing_rate=("measured_feature_missing", "mean"),
        true_label_rate=("true_label", "mean"),
        observed_label_rate=("observed_label", "mean"),
    )
    .reset_index()
)

quality_summary

## Interpretation

Missingness and label noise can vary across groups and measurement-quality levels, creating downstream bias.